# 04 — Gold Layer: Customer Spending Intelligence

This notebook reads from Silver and builds the **Gold layer** — pre-aggregated business tables that analysts can query directly without touching raw data.

We produce two Gold tables:
1. **`customer_monthly_summary`** — per customer per month KPIs (spend, income, stress ratio)
2. **`customer_profile`** — lifetime customer profile with financial stress flag

These tables are what a bank's analytics team or data scientist would actually use for reporting, segmentation, and modelling.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_PATH = 'dbfs:/FileStore/sa-banking-pipeline/silver'
GOLD_PATH   = 'dbfs:/FileStore/sa-banking-pipeline/gold'

print('Spark ready:', spark.version)

## Step 1 — Read Silver tables

In [ ]:
customers = spark.read.format('delta').load(f'{SILVER_PATH}/customers')
txn       = spark.read.format('delta').load(f'{SILVER_PATH}/transactions')

debits  = txn.filter(F.col('transaction_type') == 'DEBIT')
credits = txn.filter(F.col('transaction_type') == 'CREDIT')

print(f'Customers: {customers.count():,}')
print(f'Total transactions: {txn.count():,}')
print(f'  Debits:  {debits.count():,}')
print(f'  Credits: {credits.count():,}')

## Step 2 — Build customer_monthly_summary

For every customer, for every month, we calculate:
- How much they spent
- How much income came in
- What they spent most on
- How much of their spending happened after the 20th (month-end stress signal)

In [ ]:
# Monthly spend stats
monthly_spend = debits.groupBy('customer_id', 'txn_year', 'txn_month').agg(
    F.round(F.sum('amount'), 2).alias('total_spend'),
    F.count('transaction_id').alias('debit_txn_count'),
    F.countDistinct('merchant_name').alias('unique_merchants'),
    F.round(F.avg('amount'), 2).alias('avg_txn_amount'),
    F.round(F.max('amount'), 2).alias('max_txn_amount'),
)

# Monthly income received
monthly_income = credits.groupBy('customer_id', 'txn_year', 'txn_month').agg(
    F.round(F.sum('amount'), 2).alias('total_income_received'),
    F.count('transaction_id').alias('credit_txn_count'),
)

# Month-end spend (after day 20 — SA financial stress signal)
month_end_spend = debits.filter(F.col('is_month_end')).groupBy('customer_id', 'txn_year', 'txn_month').agg(
    F.round(F.sum('amount'), 2).alias('month_end_spend')
)

print('Monthly aggregations built')

In [ ]:
# Top spending category per customer per month
cat_spend = debits.groupBy('customer_id', 'txn_year', 'txn_month', 'merchant_category').agg(
    F.sum('amount').alias('cat_spend')
)
w = Window.partitionBy('customer_id', 'txn_year', 'txn_month').orderBy(F.desc('cat_spend'))
top_cat = cat_spend.withColumn('rank', F.rank().over(w)).filter(F.col('rank') == 1) \
    .select('customer_id', 'txn_year', 'txn_month', F.col('merchant_category').alias('top_category'))

print('Top category per month calculated')

In [ ]:
# Join everything into one monthly summary table
monthly_summary = monthly_spend \
    .join(monthly_income,   ['customer_id', 'txn_year', 'txn_month'], 'left') \
    .join(month_end_spend,  ['customer_id', 'txn_year', 'txn_month'], 'left') \
    .join(top_cat,          ['customer_id', 'txn_year', 'txn_month'], 'left') \
    .withColumn('month_end_stress_ratio',
        F.round(F.col('month_end_spend') / F.col('total_spend'), 4)) \
    .withColumn('spend_to_income_ratio',
        F.round(F.col('total_spend') / F.col('total_income_received'), 4)) \
    .withColumn('_gold_processed_at', F.current_timestamp())

print(f'Monthly summary: {monthly_summary.count():,} rows')
monthly_summary.show(5)

In [ ]:
# Write Gold — customer_monthly_summary
monthly_summary.write.format('delta').mode('overwrite') \
    .partitionBy('txn_year', 'txn_month') \
    .save(f'{GOLD_PATH}/customer_monthly_summary')
print('✅ customer_monthly_summary written to Gold')

## Step 3 — Build customer_profile

A single row per customer summarising their lifetime behaviour — income band, preferred channel, spending diversity, and whether they show signs of financial stress.

In [ ]:
# Lifetime spend stats
spend_stats = debits.groupBy('customer_id').agg(
    F.round(F.sum('amount'), 2).alias('lifetime_spend'),
    F.count('transaction_id').alias('lifetime_txn_count'),
    F.round(F.avg('amount'), 2).alias('avg_txn_value'),
    F.countDistinct('merchant_category').alias('category_diversity'),
)

# Preferred channel (most used)
channel_counts = debits.groupBy('customer_id', 'channel').agg(F.count('*').alias('cnt'))
wc = Window.partitionBy('customer_id').orderBy(F.desc('cnt'))
top_channel = channel_counts.withColumn('rank', F.rank().over(wc)).filter(F.col('rank') == 1) \
    .select('customer_id', F.col('channel').alias('preferred_channel'))

# Financial stress from monthly summary
stress = monthly_summary.groupBy('customer_id').agg(
    F.round(F.avg('month_end_stress_ratio'), 4).alias('avg_stress_ratio'),
    F.round(F.avg('spend_to_income_ratio'), 4).alias('avg_spend_to_income'),
)
stress = stress.withColumn('is_financially_stressed', F.col('avg_stress_ratio') > 0.6)

print('Profile components built')

In [ ]:
# Join into customer_profile
customer_profile = customers \
    .join(spend_stats,  'customer_id', 'left') \
    .join(top_channel,  'customer_id', 'left') \
    .join(stress,       'customer_id', 'left') \
    .withColumn('_gold_processed_at', F.current_timestamp())

print(f'Customer profile: {customer_profile.count():,} rows')
customer_profile.select('customer_id','segment','province','monthly_income',
    'lifetime_spend','preferred_channel','avg_stress_ratio','is_financially_stressed').show(10)

In [ ]:
# Write Gold — customer_profile
customer_profile.write.format('delta').mode('overwrite').save(f'{GOLD_PATH}/customer_profile')
print('✅ customer_profile written to Gold')

## ✅ Done — Full medallion pipeline complete

| Layer | Tables |
|---|---|
| Bronze | customers, transactions |
| Silver | customers, transactions (cleaned + SA flags) |
| Gold | customer_monthly_summary, customer_profile |

**Next:** Open `05_analysis.ipynb` for business insights